# 02 - Limpeza dos Dados de Segurança

Padroniza a base de ocorrências do Fogo Cruzado, normaliza os nomes dos bairros para a lista de bairros de Recife usada no projeto e salva a base limpa em `data/processed/`.

In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

In [ ]:
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / '.git').exists():
            return p
    return Path.cwd()


ROOT = find_root()
YEAR = 2025
CSV_SEPARATOR = ','


def read_csv_flex(path):
    return pd.read_csv(path, sep=None, engine='python', encoding='utf-8-sig')


INPUT_PATH = ROOT / 'data' / 'raw' / 'seguranca' / f'fogo_cruzado_recife_{YEAR}.csv'
BAIRROS_REFERENCIA_PATH = ROOT / 'data' / 'processed' / 'notas_renda.csv'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'ocorrencias_seguranca_recife.csv'
UNMATCHED_PATH = ROOT / 'data' / 'processed' / 'ocorrencias_seguranca_bairros_nao_mapeados.csv'

print(f'Entrada : {INPUT_PATH}')
print(f'Saída   : {OUTPUT_PATH}')

In [ ]:
# Carrega ocorrências brutas e a lista de bairros de Recife usada no projeto
df = read_csv_flex(INPUT_PATH)
bairros_ref = read_csv_flex(BAIRROS_REFERENCIA_PATH)['bairro'].dropna().astype(str)

print(f'Ocorrências brutas: {len(df)}')
print(f'Bairros de referência: {len(bairros_ref)}')
display(df.head(5).reset_index(drop=True).style.hide(axis='index'))
display(bairros_ref.head(10).to_frame().style.hide(axis='index'))

## 1 - Padronização de datas, horários e campos numéricos

In [ ]:
def periodo_do_dia(hour):
    if pd.isna(hour):
        return 'sem horario'
    hour = int(hour)
    if 0 <= hour < 6:
        return 'madrugada'
    if 6 <= hour < 12:
        return 'manha'
    if 12 <= hour < 18:
        return 'tarde'
    return 'noite'


df = df.drop_duplicates(subset=['id']).copy()
df['data_ocorrencia'] = pd.to_datetime(df['data_ocorrencia'], errors='coerce', utc=True)
df['ano'] = df['data_ocorrencia'].dt.year
df['mes'] = df['data_ocorrencia'].dt.month
df['hora'] = df['data_ocorrencia'].dt.hour
df['periodo'] = df['hora'].apply(periodo_do_dia)

for col in ['latitude', 'longitude', 'mortos', 'feridos', 'baleados']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print(f'Ocorrências após deduplicação: {len(df)}')
display(df[['data_ocorrencia', 'ano', 'mes', 'hora', 'periodo', 'mortos', 'feridos', 'baleados']].head(10).style.hide(axis='index'))

## 2 - Padronização dos bairros

A chave de integração com as outras dimensões é o nome do bairro. Por isso, os nomes da API são normalizados para a lista de bairros de Recife usada no projeto.

In [ ]:
ALIASES_BAIRROS = {
    'ALTO SANTA TERESINHA': 'ALTO SANTA TEREZINHA',
    'ILHA DE JOANA BEZERRA': 'ILHA JOANA BEZERRA',
    'POCO DA PANELA': 'POCO',
    'SITIO DOS PINTOS': 'SITIO DOS PINTOS SAO BRAS',
    'SITIO DOS PINTOS SAO BRAS': 'SITIO DOS PINTOS SAO BRAS',
}


def normalizar_texto(value):
    text = '' if pd.isna(value) else str(value)
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^A-Za-z0-9]+', ' ', text).strip().upper()
    return re.sub(r'\s+', ' ', text)


mapa_bairros = {normalizar_texto(bairro): bairro for bairro in bairros_ref}
for origem, destino in ALIASES_BAIRROS.items():
    destino_padrao = mapa_bairros.get(normalizar_texto(destino))
    if destino_padrao:
        mapa_bairros[normalizar_texto(origem)] = destino_padrao

df['bairro_normalizado'] = df['bairro_original'].apply(normalizar_texto)
df['bairro'] = df['bairro_normalizado'].map(mapa_bairros)
df['bairro_mapeado'] = df['bairro'].notna()

nao_mapeados = (
    df.loc[~df['bairro_mapeado'], ['bairro_original', 'bairro_normalizado']]
    .drop_duplicates()
    .sort_values('bairro_original')
)

print(f'Bairros mapeados    : {df["bairro_mapeado"].sum()} ocorrências')
print(f'Bairros não mapeados: {len(nao_mapeados)} nomes distintos')
display(nao_mapeados.reset_index(drop=True).style.hide(axis='index'))

## 3 - Salvar base limpa

In [ ]:
if not nao_mapeados.empty:
    nao_mapeados.to_csv(UNMATCHED_PATH, index=False, sep=CSV_SEPARATOR, encoding='utf-8')
    print(f'Bairros não mapeados salvos em: {UNMATCHED_PATH}')

df_limpo = df[df['bairro_mapeado']].copy()
df_limpo['motivo_principal'] = df_limpo['motivo_principal'].fillna('Nao informado')
df_limpo['ocorrencia_com_vitima'] = df_limpo['baleados'] > 0

cols_saida = [
    'id', 'documento', 'bairro', 'bairro_original', 'endereco',
    'latitude', 'longitude', 'data_ocorrencia', 'ano', 'mes', 'hora', 'periodo',
    'motivo_principal', 'motivos_complementares', 'acao_policial', 'presenca_agente', 'chacina',
    'mortos', 'feridos', 'baleados', 'civis_mortos', 'civis_feridos', 'agentes_mortos', 'agentes_feridos',
    'ocorrencia_com_vitima',
]

df_limpo[cols_saida].to_csv(OUTPUT_PATH, index=False, sep=CSV_SEPARATOR, encoding='utf-8')

print(f'Arquivo salvo: {OUTPUT_PATH}')
print(f'Ocorrências limpas: {len(df_limpo)}')
print(f'Bairros com ocorrência: {df_limpo["bairro"].nunique()}')
display(df_limpo[cols_saida].head(10).reset_index(drop=True).style.hide(axis='index'))

In [ ]:
# Conferências finais para seguir para a análise
resumo = df_limpo.groupby('bairro', as_index=False).agg(
    ocorrencias=('id', 'nunique'),
    mortos=('mortos', 'sum'),
    feridos=('feridos', 'sum'),
    baleados=('baleados', 'sum'),
).sort_values('ocorrencias', ascending=False)

display(resumo.head(15).reset_index(drop=True).style.hide(axis='index'))
display(df_limpo.isnull().sum().to_frame('qtd_nulos').style.hide(axis='index'))